# Case study: imbalance strategies for churners and rare headings

*Session 9, block 3 practice. Author: course team, licence CC-BY-4.0.*

Two natural imbalanced tasks:

- **Part A, churn.** IBM Telco customers, 26.5 % of whom left. Which customers will leave? Strategies inside imbalanced-learn pipelines (undersampling, oversampling, SMOTENC, class weights) with a logistic regression and a random forest, compared with a tuned decision threshold. Measures: precision, recall, F1 of the churn class, ROC AUC and PR AUC.
- **Part B, the long tail of headings.** All headings of the 50,000-decision EBTI sample, a linear classifier on the TF-IDF matrix of the descriptions, time-based validation (2017–2021 → 2022–2023). Strategies: none, balanced class weights, oversampling rare headings to a minimum count, grouping rare headings. Measure: macro-F1.

Theory: [page 3](../theory/03-class-imbalance.md). Part A downloads the Telco data from GitHub; Part B needs `case-study/data/train_sample.parquet`. The notebook runs in about two minutes.

In [ ]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
DATA = ROOT / "case-study" / "data"
from imblearn.over_sampling import SMOTENC, RandomOverSampler
from imblearn.pipeline import make_pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# Part A: which customers will leave?

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0)
y = (telco["Churn"] == "Yes").astype(int)
X = telco.drop(columns=["customerID", "Churn"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
print(y.value_counts().to_dict(), "churn rate", round(y.mean(), 3))
cv = StratifiedKFold(5, shuffle=True, random_state=0)

SMOTENC needs to know which columns are categorical *after* preprocessing. We therefore encode the categories as integers (ordinal) and scale the numbers first, let SMOTENC interpolate only the numeric columns, and one-hot encode afterwards for the logistic regression. The forest can work with the ordinal codes directly.

In [ ]:
ordinal = make_column_transformer((OrdinalEncoder(), cat_cols), remainder=StandardScaler())   # categories first
cat_idx = list(range(len(cat_cols)))
onehot = make_column_transformer((OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_idx),
                                 remainder="passthrough")


def strategies(model, weighted):
    return {
        "no correction": make_pipeline(ordinal, onehot, model()),
        "undersampling": make_pipeline(ordinal, RandomUnderSampler(random_state=0), onehot, model()),
        "oversampling": make_pipeline(ordinal, RandomOverSampler(random_state=0), onehot, model()),
        "SMOTENC": make_pipeline(ordinal, SMOTENC(categorical_features=cat_idx, random_state=0), onehot, model()),
        "class weights": make_pipeline(ordinal, onehot, weighted()),
    }


models = {
    "logistic regression": strategies(lambda: LogisticRegression(max_iter=2000),
                                      lambda: LogisticRegression(max_iter=2000, class_weight="balanced")),
    "random forest": strategies(lambda: RandomForestClassifier(300, min_samples_leaf=5, random_state=0, n_jobs=-1),
                                lambda: RandomForestClassifier(300, min_samples_leaf=5, class_weight="balanced",
                                                               random_state=0, n_jobs=-1)),
}


def scores(proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {"precision": precision_score(y, pred), "recall": recall_score(y, pred), "f1": f1_score(y, pred),
            "accuracy": accuracy_score(y, pred), "roc_auc": roc_auc_score(y, proba),
            "pr_auc": average_precision_score(y, proba)}


rows, probas = [], {}
for model_name, cands in models.items():
    for name, pipe in cands.items():
        proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]   # samplers: training folds only
        probas[(model_name, name)] = proba
        rows.append({"model": model_name, "strategy": name, **scores(proba)})
pd.DataFrame(rows).set_index(["model", "strategy"]).round(3)

### The alternative: keep the model, move the threshold

A threshold chosen on the cross-validated probabilities of the uncorrected model. Here we pick the threshold with the best F1, for illustration; in practice choose it from costs (Session 8) and on validation data only.

In [ ]:
for model_name in models:
    proba = probas[(model_name, "no correction")]
    prec, rec, thr = precision_recall_curve(y, proba)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
    best = thr[np.argmax(f1[:-1])]
    print(model_name, "best-F1 threshold", round(best, 2), {k: round(v, 3) for k, v in scores(proba, best).items()})

**Questions.**

1. Which strategies change ROC AUC and PR AUC, and which only move precision and recall? What does that tell you?
2. Compare the best correction with the uncorrected model at its best threshold. Is resampling worth the extra step?
3. A retention team can call 1,000 customers a month. Which number matters to them: F1, recall, or the precision among the 1,000 customers with the highest predicted risk? Compute the last one for two strategies.

In [ ]:
# precision among the 1,000 customers with the highest predicted churn risk
for key in [("logistic regression", "no correction"), ("logistic regression", "SMOTENC")]:
    top = np.argsort(-probas[key])[:1000]
    print(key, "precision in top 1,000:", round(y.iloc[top].mean(), 3))

# Part B: the long tail of headings

In [ ]:
decisions = pd.read_parquet(DATA / "train_sample.parquet")
fit = (decisions["start_date"] < "2022-01-01").to_numpy()
val = ~fit
y = decisions["heading"].to_numpy()
counts_fit = Counter(y[fit])
print("headings in the fitting rows:", len(counts_fit),
      "| with fewer than 5 rows:", sum(n < 5 for n in counts_fit.values()))
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_fit = tfidf.fit_transform(decisions.loc[fit, "description"])
X_val = tfidf.transform(decisions.loc[val, "description"])


def linear(class_weight=None):
    return SGDClassifier(loss="hinge", alpha=1e-5, class_weight=class_weight, random_state=0, n_jobs=-1)

In [ ]:
rows = []


def record(name, pred):
    rows.append({"strategy": name, "accuracy": accuracy_score(y[val], pred),
                 "macro-F1": f1_score(y[val], pred, average="macro", zero_division=0),
                 "headings predicted": len(set(pred))})


record("no correction", linear().fit(X_fit, y[fit]).predict(X_val))
record("balanced class weights", linear("balanced").fit(X_fit, y[fit]).predict(X_val))
for minimum in (20, 50):
    sampler = RandomOverSampler(sampling_strategy={c: max(n, minimum) for c, n in counts_fit.items()},
                                random_state=0)
    X_res, y_res = sampler.fit_resample(X_fit, y[fit])          # fitting rows only
    record(f"oversample to >= {minimum}", linear().fit(X_res, y_res).predict(X_val))
grouped = np.where([counts_fit[h] < 5 for h in y[fit]], "rare", y[fit])
record("group headings < 5 rows", linear().fit(X_fit, grouped).predict(X_val))
pd.DataFrame(rows).set_index("strategy").round(3)

**Questions.** Which strategy changes macro-F1 most, and in which direction? Why can a model that predicts the class "rare" never be right on the leaderboard?

## Exercises

1. In Part A, compare `RandomOverSampler(sampling_strategy=0.5)` (churners raised to half the loyal customers) with full balancing.
2. In Part B, try `sampling_strategy` minimum counts of 5 and 100. Is there a best value?
3. Both parts: write one sentence per strategy in the form "recall of … rose from … to …, precision fell from … to …".